In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import os
os.chdir("/content/gdrive/MyDrive/LLM/CBLLMMODEL")

In [ ]:
import pandas as pd

In [ ]:
Df = pd.read_excel("CBRT_DecisionReportsV1.xlsx", index_col="REPORTNAME")
Df.head(3)

In [ ]:
import nltk
nltk.download('punkt')  # ilk kez kullanıyorsan
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

Sentence_Df = pd.DataFrame(index = range(1,5), columns = ["Date", "Sentences"])

index = 1
for row in Df.index:

    text = Df.loc[row, "Text"]
    date = Df.loc[row, "Date"]
    sentences = sent_tokenize(text)


    for sentence in sentences:
        Sentence_Df.at[index, "Date"] = date
        Sentence_Df.at[index, "Sentences"] = sentence

        index+=1

Sentence_Df.to_excel("CBRT_DecisionReportsV2.xlsx")
Sentence_Df.head(3)

In [ ]:
from transformers import pipeline
import pandas as pd
hd_classifier = pipeline("text-classification", model="mrince/CBRT-RoBERTa-HawkishDovish-Classifier")
agent_classifier = pipeline("text-classification", model="Moritz-Pfeifer/CentralBankRoBERTa-agent-classifier")
sentiment_classifier = pipeline("text-classification", model="Moritz-Pfeifer/CentralBankRoBERTa-sentiment-classifier")

In [ ]:
Analysis_Df = pd.read_excel("CBRT_DecisionReportsV2.xlsx", index_col = "Unnamed: 0")
Analysis_Df["Hawkish/Dovish"] = None
Analysis_Df["Sentiment"] = None
Analysis_Df["Agent"] = None

Analysis_Df.head(3)

In [ ]:
hd_classifier("On the other hand, the recent deceleration in economic activity may curb services inflation.")
agent_classifier("On the other hand, the recent deceleration in economic activity may curb services inflation.")
hd_classifier("On the other hand, the recent deceleration in economic activity may curb services inflation.")

In [ ]:
for row in Analysis_Df.index:
    sentence = Analysis_Df.loc[row, "Sentences"]

    hk_label = hd_classifier(sentence)[0]["label"]
    sentiment = sentiment_classifier(sentence)[0]["label"]
    agent = agent_classifier(sentence)[0]["label"]

    if hk_label == "LABEL_0":
        hk = "Neutral"
    elif hk_label == "LABEL_1":
        hk = "Hawkish"
    elif hk_label == "LABEL_2":
        hk = "Dovish"

    Analysis_Df.loc[row, "Hawkish/Dovish"] = hk
    Analysis_Df.loc[row, "Sentiment"] = sentiment
    Analysis_Df.loc[row, "Agent"] = agent
    if row %10 == 0:
        print(row)

Analysis_Df.to_excel("CBRT_DecisionReportsV3.xlsx")
print(Analysis_Df.head(3))

In [ ]:
Labelled_Df = pd.read_excel("CBRT_DecisionReportsV3.xlsx", index_col = "Unnamed: 0")
Labelled_Df.head()